In [0]:
# Importação das bibliotecas necessárias

from dotenv import load_dotenv
from pathlib import Path
import os

from pyspark.sql import functions as F

In [0]:
# Localiza e carrega o arquivo .env com as configurações

env_path = next(Path("/Workspace").rglob(".env"), None)

if env_path:
    load_dotenv(env_path)
    print(".env carregado com sucesso.")
else:
    print(".env não encontrado.")

sql_host = os.getenv("SQL_HOST")
sql_database = os.getenv("SQL_DATABASE")
sql_username = os.getenv("SQL_USERNAME")
sql_password = os.getenv("SQL_PASSWORD")

tabela_destino = "squad2.ecommerce_clientes"

print(f"Tabela de análise: {tabela_destino}")

.env carregado com sucesso.
Tabela de análise: squad2.ecommerce_clientes


In [0]:
# Lê a tabela de clientes diretamente do SQL Server

df_clientes_analise = spark.read \
    .format("sqlserver") \
    .option("host", sql_host) \
    .option("port", 1433) \
    .option("database", sql_database) \
    .option("dbtable", tabela_destino) \
    .option("user", sql_username) \
    .option("password", sql_password) \
    .load()

print("Tabela lida do SQL Server com sucesso.")

Tabela lida do SQL Server com sucesso.


In [0]:
# Validação da estrutura e amostragem dos dados

print("=== ESTRUTURA DOS DADOS ===")
df_clientes_analise.printSchema()

print("\n=== AMOSTRA DOS DADOS ===")
display(df_clientes_analise.limit(10))

=== ESTRUTURA DOS DADOS ===
root
 |-- id_cliente: long (nullable = true)
 |-- uuid_cliente: string (nullable = true)
 |-- nome: string (nullable = true)
 |-- sobrenome: string (nullable = true)
 |-- email: string (nullable = true)
 |-- senha_hash: string (nullable = true)
 |-- dt_cadastro: timestamp (nullable = true)
 |-- dt_ultima_atualizacao: timestamp (nullable = true)


=== AMOSTRA DOS DADOS ===


id_cliente,uuid_cliente,nome,sobrenome,email,senha_hash,dt_cadastro,dt_ultima_atualizacao
1790000559,d8368d34-54c8-4348-98ed-6e316f7fbf8d,Maria Liz,Cunha,maria.cunha49@gmail.com,3f3d00a645be06953fc69e8c0252a5f669c802d02979d22d6dc3d7b8a3cdfe85,2023-10-27T15:44:38.000Z,2024-05-06T15:44:38.000Z
1790001000,a81889bc-1f16-4f8c-94ce-2e863b97623c,Erick,Moreira Cardoso,cardoso.erick1973@gmail.com,897d47a899e11d88d5a585bf7c7d6ed45dd60d79769ff68b1973b685d9c5a3ae,2025-06-23T16:55:26.000Z,2026-04-02T16:55:26.000Z
1790000760,760cc2f6-6df8-4bf0-8a1c-04c3971bf938,Maria,Martins Souza,souza.maria@gmail.com,e9dc2c7f75e28661fe6324095fbf00a0449393587663ab08cda0bea719e6b397,2023-10-10T03:01:56.000Z,2024-06-03T03:01:56.000Z
1790000882,423828b5-eae5-4b87-8f60-219de0c97259,Nina,Andrade de Lima,nina_lima@gmail.com,88c72ca36ca652efc00416a97733cc108f244c02ccabdf9084cc18dabc3dadb0,2026-08-28T11:56:56.000Z,2027-08-24T11:56:56.000Z
1790000560,03d114bd-0b54-427a-907f-3cea12182590,Maria Júlia,Viana de Pacheco,mariapacheco@gmail.com,d24f203919184a6ac70017b51f7c88c76f9aace342d01ae4f53861e41b7e8eeb,2024-01-22T00:24:03.000Z,2024-12-14T00:24:03.000Z
1790000561,094503e1-8080-4a4f-9fb7-e23aae32c1d8,Maria Laura,da Costa Caldeira,mariacaldeira@hotmail.com,1e9ed4ef0b3b887cd43e7b865fd096670268ece73d8e41210e4a4052deb75055,2026-09-24T11:22:39.477Z,2027-07-03T11:22:39.477Z
1790000880,5c948a18-2e41-46e9-add9-fc5edbd82499,Thomas,da Rocha Macedo,thomasm1996@hotmail.com,bafed162375090da362bf7fcadc0ebeb098ca5949fe10e293c01c1428c3a3b6e,2024-02-22T00:24:10.000Z,2024-12-12T00:24:10.000Z
1790000881,2826d96e-0c6c-4cef-89a1-7e882d6fb3c2,Maria Fernanda,Freitas do Gomes,mgomes1996@outlook.com,49a0c770f52d39a4d2942c55cce6093e28633518dae4d3c2622968c8b5f6e226,2024-09-11T22:40:49.000Z,2024-09-23T22:40:49.000Z
1790001002,a3e888ab-d827-4516-805d-e7698abc670a,Luiz Felipe,Nogueira da Cunha,luiz.c6@gmail.com,3f8c80fb1c68df6ffd3d20a39a6d634820c63d9a7b28d3011dd62fd7463c0c5e,2024-10-30T05:27:34.000Z,2025-08-18T05:27:34.000Z
1790000640,ac09fb55-a807-4cf3-9419-0d60520127f7,Luiz Gustavo,Campos Vieira,luiz.vieira@gmail.com,550654aececdbcb3067878e6ebf215e37d267a613e19906f92c1bd14d35054e2,2024-08-08T13:01:57.000Z,2025-01-14T13:01:57.000Z


In [0]:
# Validação do volume de dados

quantidade_linhas = df_clientes_analise.count()
quantidade_colunas = len(df_clientes_analise.columns)

print("=== VOLUME DE DADOS ===")
print(f"Linhas   : {quantidade_linhas}")
print(f"Colunas  : {quantidade_colunas}")

=== VOLUME DE DADOS ===
Linhas   : 13
Colunas  : 8


In [0]:
# Validação da chave primária id_cliente

nulos_id_cliente = df_clientes_analise.filter(
    F.col("id_cliente").isNull()
).count()

print("=== INTEGRIDADE DA CHAVE PRIMÁRIA ===")
print(f"Registros com id_cliente nulo: {nulos_id_cliente}")

if nulos_id_cliente == 0:
    print("Status: saudável - nenhum registro com id_cliente nulo.")
else:
    print("Status: atenção - existem registros com id_cliente nulo.")

=== INTEGRIDADE DA CHAVE PRIMÁRIA ===
Registros com id_cliente nulo: 0
Status: saudável - nenhum registro com id_cliente nulo.
